# Conway's Game of Life as a convolution

*others · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/game_of_life)

Not physics, just for fun: a grid of live and dead cells where every cell looks at its eight neighbours. Counting them is a $3\times3$ convolution, written as one einsum over unfolded patches, and the birth/survival rule is a comparison of tensors, so every cell of several boards updates at once. Watch gliders, a glider gun and random soups.

**Vector form**

$$N(\mathbf r) = \sum_{\mathbf d \in \text{8 neighbours}} X(\mathbf r + \mathbf d),\qquad X'(\mathbf r) = \begin{cases} 1 & N = 3 \\ X(\mathbf r) & N = 2 \\ 0 & \text{otherwise} \end{cases}$$

**Index (tensor) form**

$$N_{bp} = K_k\,P_{bkp},\qquad P_{bkp} = X_{b,\,i_p + a_k - 1,\,j_p + b_k - 1},\qquad X'_{bp} = [N_{bp} = 3] + X_{bp}\,[N_{bp} = 2]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $X_{bij}$ | `board(pattern, size, ...)` | (B, H, W) | cell $(i, j)$ of board $b$: 1 alive, 0 dead |
| $K_{ab}$ | `kernel(device)` | (3, 3) | ones around the cell and 0 in the middle: a cell is not its own neighbour |
| $P_{bkp}$ | `F.unfold(x[:, None], 3, padding=1)` | (B, 9, H·W) | the $3\times3$ patch around every cell $p$ laid out as a column; cells outside the board count as dead |
| $N_{bp} = K_k P_{bkp}$ | `einsum('k,bkp->bp', K.flatten(), P)` | (B, H·W) | repeated $k$ ⇒ summed: live neighbours of every cell, the same as a convolution with $K$ |
| $X'_{bp}$ | `(N == 3) \| (X & (N == 2))` | (B, H, W) | B3/S23: born with exactly 3 neighbours, survives with 2 or 3, otherwise dies |
| $\sum_{ij} X_{bij}$ | `history.sum(dim=(-2, -1))` | (G+1, B) | population of every board in every generation |

Three numbers (born with 3, survive with 2 or 3) are enough for gliders that walk diagonally one cell every 4 generations, oscillators such as the pulsar (period 3) and the Gosper glider gun, which fires a new glider every 30 generations so its population grows by 5 each period. The $R$-pentomino, only 5 cells, keeps changing for 1103 generations on an unbounded grid; on a finite board the edge cuts this short. Random soups burn out into still lifes, blinkers and escaping gliders; the grey curves show how similar this decay is from soup to soup. The same unfold-and-einsum pattern is how a convolutional neural network layer is computed.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/game_of_life/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import neighbours


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k = tensors.kernel(dev)  # (3, 3)
    x0 = tensors.board(params["pattern"], params["size"], params["n_boards"], params["density"], 0, dev)  # (B, H, W)
    print(f"pattern {params['pattern']}: {x0.shape[0]} board(s) of {x0.shape[1]}×{x0.shape[2]} cells")
    print(f"neighbour counts at generation 0 range over {sorted(neighbours(x0, k).unique().int().tolist())}")

    history = solve.evolve(x0, k, params["generations"])  # (G+1, B, H, W)
    pop = solve.population(history)  # (G+1, B)
    for g in (0, len(pop) // 4, len(pop) // 2, len(pop) - 1):
        print(f"generation {g:4d}: population {pop[g].tolist()}")
    repeat = solve.first_repeat(history)
    if repeat is None:
        print("board 0 has not repeated yet: still evolving")
    else:
        print(f"board 0 repeats an earlier state at generation {repeat}: only still lifes and oscillators remain")
    return plot.animate(history, pop, params["pattern"], out_dir / "game_of_life.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/game_of_life/equation.py
r"""Rules → tensor translation.

Conway's Game of Life is a cellular automaton, not physics, but it is the purest example of a local update rule
written as a tensor operation. Each cell X_{ij} ∈ {0, 1} counts its eight live neighbours,

    N_{ij} = \sum_{a,b} K_{ab}\,X_{i+a-1,\,j+b-1},       K = [[1, 1, 1], [1, 0, 1], [1, 1, 1]],

which is a convolution. Writing it as one einsum: `unfold` copies the 3×3 patch around every cell into a column,
P_{bkp} (k = patch position, p = cell), and the count is N_{bp} = K_k P_{bkp}. Then all cells update at once (B3/S23):

    X'_{ij} = [N_{ij} = 3] + X_{ij}\,[N_{ij} = 2]

— a dead cell with exactly 3 neighbours is born, a live cell with 2 or 3 survives, every other cell dies.
Cells outside the board are dead.
"""

import torch
import torch.nn.functional as F


def neighbours(x: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    r"""N_{bp} = K_k P_{bkp}   (k summed over the 9 patch positions).   x: (B, H, W) → (B, H, W)"""
    b, h, w = x.shape
    patches = F.unfold(x[:, None], kernel_size=3, padding=1)  # (B, 9, H·W)
    return torch.einsum("k,bkp->bp", k.flatten(), patches).reshape(b, h, w)


def step(x: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    r"""X' = [N = 3] + X·[N = 2]: one generation for every board.   (B, H, W) → (B, H, W)"""
    n = neighbours(x, k)
    return ((n == 3) | ((x > 0) & (n == 2))).float()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/game_of_life/tensors.py
"""Build the concrete tensors: the neighbour kernel and the starting boards (classic patterns or random soups)."""

import re

import torch

# Classic patterns in run-length encoding (RLE): b = dead, o = alive, $ = next row, a number repeats the next item.
PATTERNS = {
    "glider": "bo$2bo$3o!",
    "gosper_gun": (
        "24bo$22bobo$12b2o6b2o12b2o$11bo3bo4b2o12b2o$2o8bo5bo3b2o$2o8bo3bob2o4bobo$10bo5bo7bo$11bo3bo$12b2o!"
    ),
    "r_pentomino": "b2o$2o$bo!",
    "pulsar": "2b3o3b3o2$o4bobo4bo$o4bobo4bo$o4bobo4bo$2b3o3b3o2$2b3o3b3o$o4bobo4bo$o4bobo4bo$o4bobo4bo2$2b3o3b3o!",
}


def kernel(device: torch.device) -> torch.Tensor:
    """K_{ab}: ones on the 3×3 neighbourhood with a zero in the middle (a cell is not its own neighbour): (3, 3)."""
    k = torch.ones(3, 3, device=device)
    k[1, 1] = 0
    return k


def parse_rle(rle: str) -> torch.Tensor:
    """Decode an RLE string into a 0/1 tensor of shape (rows, cols)."""
    rows, row = [], []
    for count, tag in re.findall(r"(\d*)([bo$!])", rle):
        n = int(count) if count else 1
        if tag in "bo":
            row += [1.0 if tag == "o" else 0.0] * n
        else:  # "$" ends a row (n − 1 extra empty rows), "!" ends the pattern
            rows.append(row)
            rows += [[] for _ in range(n - 1)] if tag == "$" else []
            row = []
    width = max(len(r) for r in rows)
    return torch.tensor([r + [0.0] * (width - len(r)) for r in rows])


def board(pattern: str, size: int, n_boards: int, density: float, seed: int, device: torch.device) -> torch.Tensor:
    """Starting boards X_{bij} of shape (B, size, size); B = n_boards random soups, or 1 board with a pattern."""
    if pattern == "random":
        gen = torch.Generator().manual_seed(seed)
        return (torch.rand(n_boards, size, size, generator=gen) < density).float().to(device)
    cells = parse_rle(PATTERNS[pattern])
    x = torch.zeros(1, size, size)
    r0 = 2 if pattern == "gosper_gun" else (size - cells.shape[0]) // 2  # the gun sits top-left, firing down-right
    c0 = 2 if pattern == "gosper_gun" else (size - cells.shape[1]) // 2
    x[0, r0 : r0 + cells.shape[0], c0 : c0 + cells.shape[1]] = cells
    return x.to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/game_of_life/solve.py
"""Run the automaton for many generations and record the boards and their populations."""

import torch

from problems._common.progress import steps

from .equation import step


def evolve(x0: torch.Tensor, k: torch.Tensor, generations: int) -> torch.Tensor:
    """History of all boards, shape (G+1, B, H, W), stored as uint8 to save memory."""
    history = torch.empty((generations + 1, *x0.shape), dtype=torch.uint8, device=x0.device)
    x = x0
    history[0] = x.to(torch.uint8)
    for g in steps(generations, "generations"):
        x = step(x, k)
        history[g + 1] = x.to(torch.uint8)
    return history


def population(history: torch.Tensor) -> torch.Tensor:
    """Live cells per board and generation: (G+1, B, H, W) → (G+1, B)."""
    return history.sum(dim=(-2, -1))


def first_repeat(history: torch.Tensor) -> int | None:
    """Generation at which board 0 first repeats an earlier state (a still life or oscillator), or None."""
    seen: dict[bytes, int] = {}
    for g, board in enumerate(history[:, 0]):
        key = board.cpu().numpy().tobytes()
        if key in seen:
            return g
        seen[key] = g
    return None


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/game_of_life/plot.py
"""Animate board 0 next to the population curves of every board."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif

CMAP = ListedColormap(["#ffffff", "#2563eb"])


def animate(history: torch.Tensor, pop: torch.Tensor, pattern: str, path: Path) -> Path:
    """history (G+1, B, H, W) uint8, pop (G+1, B)."""
    boards, counts = history[:, 0].cpu().numpy(), pop.cpu().numpy()
    gens = np.arange(len(counts))

    fig, (ax_b, ax_p) = plt.subplots(1, 2, figsize=(9, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
    image = ax_b.imshow(boards[0], cmap=CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax_b.set_xticks([])
    ax_b.set_yticks([])
    for side in ax_b.spines.values():
        side.set_color("#9ca3af")
    title = ax_b.set_title("", fontsize=10)

    for b in range(counts.shape[1]):
        ax_p.plot(
            gens,
            counts[:, b],
            lw=1.2 if b == 0 else 0.8,
            color="#2563eb" if b == 0 else "#9ca3af",
            alpha=1 if b == 0 else 0.6,
        )
    (dot,) = ax_p.plot([], [], "o", color="#2563eb", ms=6)
    ax_p.set_xlabel("generation")
    ax_p.set_ylabel("live cells")
    ax_p.set_title("population (blue: the board on the left)", fontsize=10)
    ax_p.grid(True, alpha=0.2)

    def update(g: int):
        image.set_data(boards[g])
        title.set_text(f"{pattern.replace('_', ' ')} — generation {g}")
        dot.set_data([g], [counts[g, 0]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(gens), 120), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'pattern': 'gosper_gun',
    'size': 64,
    'generations': 240,
    'n_boards': 6,
    'density': 0.3
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.game_of_life")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.game_of_life.main")
problem_dir = Path("problems/others/game_of_life")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
